# 1.5 Transfer learning: gėlių klasifikavimas su MobileNetV2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbaranovskij/krepselisai/blob/main/notebooks/1_5_transfer_learning_geles.ipynb)

**Komanda:** Edvin Vilkanec, Gabriel Baranovskij · VILNIUSTECH

Dėstytojo *Transfer learning 1* ir *Transfer learning 2* pavyzdžius perdarome su tais pačiais gėlių duomenimis kaip 1.4 užduotyje, kad matytųsi skirtumas su CNN nuo nulio.

| Etapas | Kas daroma |
|---|---|
| **1. Požymių išskyrimas** (*Transfer learning 1*) | paimamas ImageNet milijonu nuotraukų išmokytas **MobileNetV2**, jo svoriai užšaldomi, mokomas tik naujas klasifikatorius viršuje |
| **2. Fine-tuning** (*Transfer learning 2*) | atšildomi viršutiniai MobileNetV2 sluoksniai ir mokomi labai mažu greičiu |

> Colab: *Runtime → Change runtime type → GPU*, tada *Run all*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Duomenys

In [ ]:
import pathlib, tarfile, urllib.request

URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
if not pathlib.Path("flower_photos").exists():
    urllib.request.urlretrieve(URL, "flower_photos.tgz")
    tarfile.open("flower_photos.tgz").extractall(".")
data_dir = pathlib.Path("flower_photos")

for d in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    print(f"{d.name:12s} {len(list(d.glob('*.jpg')))} nuotr.")

In [ ]:
IMG, BATCH = 160, 32
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="both", seed=123, image_size=(IMG, IMG), batch_size=BATCH)
klases = train_ds.class_names
train_ds = train_ds.cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
print(klases)

## 2. Bazinis modelis: MobileNetV2

`include_top=False` reiškia, kad paimame MobileNetV2 be jo paskutinio ImageNet klasifikavimo sluoksnio (1000 klasių). Liekanti dalis iš nuotraukos padaro **5×5×1280 požymių žemėlapį**. `trainable = False` užšaldo visus jo svorius.

In [ ]:
bazinis = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
bazinis.trainable = False
print("Bazinio modelio sluoksnių:", len(bazinis.layers))

augmentacija = tf.keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.1)])

ivestis = tf.keras.Input(shape=(IMG, IMG, 3))
x = augmentacija(ivestis)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)   # pikseliai 0–255 → −1…1, kaip mokant ImageNet
x = bazinis(x, training=False)                               # BatchNorm lieka „inference“ režime
x = layers.GlobalAveragePooling2D()(x)                       # 5×5×1280 → 1280
x = layers.Dropout(0.2)(x)
isvestis = layers.Dense(len(klases))(x)
modelis = tf.keras.Model(ivestis, isvestis)

modelis.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
print("Mokomų parametrų:", sum(int(np.prod(w.shape)) for w in modelis.trainable_weights))

## 3. Etapas 1: požymių išskyrimas (mokomas tik naujas klasifikatorius)

In [ ]:
E1 = 5
ist1 = modelis.fit(train_ds, validation_data=val_ds, epochs=E1)
print(f"Po 1 etapo validacijos tikslumas: {ist1.history['val_accuracy'][-1]:.3f}")

## 4. Etapas 2: fine-tuning

Atšildome MobileNetV2 sluoksnius nuo 100-ojo iki galo. Mokymosi greitis **100 kartų mažesnis** (1e-5), kad nesugadintume ImageNet išmoktų svorių, o tik juos šiek tiek pritaikytume gėlėms.

In [ ]:
bazinis.trainable = True
for sl in bazinis.layers[:100]:
    sl.trainable = False

modelis.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
print("Mokomų parametrų dabar:", sum(int(np.prod(w.shape)) for w in modelis.trainable_weights))

E2 = 5
ist2 = modelis.fit(train_ds, validation_data=val_ds, epochs=E1 + E2, initial_epoch=E1)

In [ ]:
acc = ist1.history["accuracy"] + ist2.history["accuracy"]
val = ist1.history["val_accuracy"] + ist2.history["val_accuracy"]
plt.figure(figsize=(9, 3.8))
plt.plot(acc, label="mokymas"); plt.plot(val, label="validacija")
plt.axvline(E1 - 0.5, color="gray", ls="--"); plt.text(E1 - 0.4, min(val), " fine-tuning pradžia", color="gray")
plt.title("Tikslumas: 1 etapas + fine-tuning"); plt.xlabel("epocha"); plt.legend(); plt.tight_layout(); plt.show()

_, galutinis = modelis.evaluate(val_ds, verbose=0)
print(f"Galutinis validacijos tikslumas: {galutinis:.3f}")

## 5. Prognozės

In [ ]:
x, y = next(iter(val_ds))
p = tf.nn.softmax(modelis.predict(x, verbose=0)).numpy()
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1); plt.imshow(x[i].numpy().astype("uint8")); plt.axis("off")
    ok = p[i].argmax() == y[i]
    plt.title(f"{klases[p[i].argmax()]} {p[i].max():.0%}", color="green" if ok else "red")
plt.suptitle("Prognozės: žalia – teisinga, raudona – klaida"); plt.tight_layout(); plt.show()

## 6. Išvados

- Vien su užšaldytu MobileNetV2 ir nauju klasifikatoriumi per 5 epochas gaunama apie **85–90 %** tikslumo; CNN nuo nulio (1.4) pasiekė apie 65–75 %.
- **Fine-tuning** dar pakelia tikslumą keliais procentais, nes viršutiniai sluoksniai prisitaiko prie gėlių.
- Transfer learning leidžia gauti gerą rezultatą su **mažai duomenų** ir per **trumpą laiką**, nes bazinis modelis jau moka atpažinti kraštus, tekstūras ir formas.